In [ ]:
from pathlib import Path
import json, random, sys, time, os

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.wajib.rnn.RNN import RNNScratch
from src.wajib.lstm.LSTM import LSTMScratch, ImageCaptioningPipeline
from src.wajib.shared.layers import EmbeddingLayer, DenseLayer
from src.wajib.shared.preprocessing import (
    loadFlickr8kCaptions, loadVocabulary,
)
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
FEATURES_NPY  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_features.npy"
FEATURES_IDX  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_index.json"
CAPTIONS_FILE = PROJECT_ROOT / "data/flickr8k/captions.txt"
VOCAB_PATH    = PROJECT_ROOT / "src/wajib/weights/vocab.json"
DATA_DIR      = PROJECT_ROOT / "data/flickr8k"
FLICKR_DIR    = DATA_DIR / "Images"

RNN_WEIGHTS_DIR   = PROJECT_ROOT / "src/wajib/weights/rnn"
LSTM_WEIGHTS_DIR  = PROJECT_ROOT / "src/wajib/weights/lstm"
CNN_ENCODER_PATH  = PROJECT_ROOT / "src/wajib/weights/cnn"

EMBED_DIM    = 256
MAX_LEN      = 30
CNN_FEAT_DIM = 2048
TARGET_SIZE  = (299, 299)

features_matrix = np.load(FEATURES_NPY)
with open(FEATURES_IDX) as f:
    idxNames = json.load(f)

imageFeatures = {name: features_matrix[i] for i, name in enumerate(idxNames)}
print(f"Fitur: {len(imageFeatures)} gambar")

captionsDict = loadFlickr8kCaptions(str(CAPTIONS_FILE))
vocab = loadVocabulary(str(VOCAB_PATH))
id2word = {v: k for k, v in vocab.items()}
print(f"Caption: {len(captionsDict)} gambar")
print(f"Kosa kata: {len(vocab)} token")

# Bagi ulang data (sama seperti training)
allImages = list(captionsDict.keys())
random.seed(SEED)
random.shuffle(allImages)

trainImgs = set(allImages[:6000])
valImgs   = set(allImages[6000:7000])
testImgs  = set(allImages[7000:])

testCaps = {k: v for k, v in captionsDict.items() if k in testImgs}
print(f"Data uji: {len(testCaps)} gambar")

In [ ]:
def loadCNNEncoder():
    from tensorflow.keras.applications.inception_v3 import InceptionV3
    from tensorflow.keras.layers import GlobalAveragePooling2D
    
    cnn = InceptionV3(include_top=False, weights='imagenet', input_shape=(299, 299, 3))
    cnn.trainable = False
    
    model = tf.keras.Sequential([
        cnn,
        GlobalAveragePooling2D(),
    ])
    return model


def buildRNNPipeline(rnnModelPath, numLayers):
    model = tf.keras.models.load_model(str(rnnModelPath))
    embed = EmbeddingLayer()
    embed.loadWeights(model.get_layer('embedding'))
    proj = DenseLayer()
    proj.loadWeights(model.get_layer('cnn_proj'))
    
    out = DenseLayer(activation='softmax')
    out.loadWeights(model.get_layer('output'))
    
    rnn = RNNScratch()
    rnn.loadWeights([model.get_layer(f'rnn_{i}') for i in range(numLayers)])
    
    class RNNPipeline:
        def __init__(self, rnn, proj, embed, out, vocab, id2word):
            self.rnn = rnn
            self.proj = proj
            self.embed = embed
            self.out = out
            self.vocab = vocab
            self.id2word = id2word
        
        def generateCaption(self, feat):
            from src.wajib.shared.decoder import greedyDecode
            hyp = greedyDecode(self.rnn, self.proj, self.embed, self.out, feat, self.vocab, MAX_LEN)
            return ' '.join(hyp)
    
    return RNNPipeline(rnn, proj, embed, out, vocab, id2word)


rnnModels = sorted(RNN_WEIGHTS_DIR.glob("rnn_*.keras"))
lstmModels = sorted(LSTM_WEIGHTS_DIR.glob("lstm_*.keras"))

if not rnnModels or not lstmModels:
    print("Gagal: Model RNN atau LSTM belum ada!")
    print(f"Model RNN: {rnnModels}")
    print(f"Model LSTM: {lstmModels}")
else:
    print(f"Ditemukan {len(rnnModels)} model RNN dan {len(lstmModels)} model LSTM")
    
    print("\nMemuat CNN encoder...")
    cnnEncoder = loadCNNEncoder()
    print("CNN encoder siap")
    
    print("\n" + "="*70)
    print("MENGUJI 12 VARIASI DECODER")
    print("="*70)

Notebook ini membangun pipeline lengkap image-to-caption dengan:
1. **CNN Encoder** (InceptionV3 beku): fitur visual 2048-dim
2. **Decoder RNN/LSTM**: fitur + token caption -> prediksi token berikutnya
3. **Greedy decoding**: memilih token dengan probabilitas tertinggi tiap langkah

Total 12 variasi decoder (6 RNN + 6 LSTM) diuji pada data uji (1000 gambar, 5 referensi per gambar).

**Catatan:**
- InceptionV3 + global average pooling adalah setup standar untuk captioning
- Model RNN dan LSTM dilatih terpisah di Notebook 3 dan 4
- Metrik evaluasi: BLEU-4 (overlap n-gram) dan METEOR (kemiripan semantik)

In [ ]:
allResults = {}
for modelPath in rnnModels:
    name = modelPath.stem
    numLayers = int(name.split('L')[0].split('_')[1])
    
    try:
        rnnPipeline = buildRNNPipeline(modelPath, numLayers)
        hypotheses = []
        references = []
        t0 = time.time()
        
        for i, (imgName, caps) in enumerate(testCaps.items()):
            if i % 200 == 0:
                print(f"  {i}/{len(testCaps)}...", end='\r')
            
            if imgName not in imageFeatures:
                continue
            
            feat = imageFeatures[imgName]
            hyp = rnnPipeline.generateCaption(feat)
            hypotheses.append(hyp.split())
            references.append([cap.split() for cap in caps])
        
        elapsed = time.time() - t0
        
        bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))
        
        meteorScores = []
        for refs, hyp in zip(references, hypotheses):
            meteorScores.append(max([meteor_score(ref, hyp) for ref in refs]))
        meteor = np.mean(meteorScores)
        
        allResults[name] = {
            'bleu4': bleu4,
            'meteor': meteor,
            'time_s': elapsed,
            'type': 'RNN',
        }
        print(f"{name:<22} BLEU-4={bleu4:.4f} METEOR={meteor:.4f} waktu={elapsed:.1f}s")
        
    except Exception as e:
        print(f"Gagal di {name}: {e}")

for modelPath in lstmModels:
    name = modelPath.stem
    numLayers = int(name.split('L')[0].split('_')[1])
    
    try:
        model = tf.keras.models.load_model(str(modelPath))
        
        embed = EmbeddingLayer()
        embed.loadWeights(model.get_layer('embedding'))
        
        proj = DenseLayer()
        proj.loadWeights(model.get_layer('cnn_proj'))
        
        out = DenseLayer(activation='softmax')
        out.loadWeights(model.get_layer('output'))
        
        lstm = LSTMScratch()
        lstm.loadWeights([model.get_layer(f'lstm_{i}') for i in range(numLayers)])
        
        pipeline = ImageCaptioningPipeline(
            cnnEncoder=cnnEncoder,
            projectDense=proj,
            embeddingLayer=embed,
            lstmScratch=lstm,
            outputDense=out,
            vocab=vocab,
            id2word=id2word,
            targetSize=TARGET_SIZE,
            maxLen=MAX_LEN,
        )
        
        hypotheses = []
        references = []
        t0 = time.time()
        
        for i, (imgName, caps) in enumerate(testCaps.items()):
            if i % 200 == 0:
                print(f"  {i}/{len(testCaps)}...", end='\r')
            
            if imgName not in imageFeatures:
                continue
            
            feat = imageFeatures[imgName]
            
            x = proj.forward(feat)
            h = [np.zeros(cell.hiddenDim) for cell in lstm.cells]
            c = [np.zeros(cell.hiddenDim) for cell in lstm.cells]
            
            words = []
            token = vocab['<start>']
            
            for _ in range(MAX_LEN):
                x_emb = embed.forward(token)
                h, c = lstm.forwardStep(x_emb, h, c)
                logits = out.forward(h[-1])
                token = int(np.argmax(logits))
                
                if token == vocab['<end>']:
                    break
                words.append(id2word.get(token, '<unk>'))
            
            hyp = ' '.join(words)
            hypotheses.append(hyp.split())
            references.append([cap.split() for cap in caps])
        
        elapsed = time.time() - t0
        
        bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))
        
        meteorScores = []
        for refs, hyp in zip(references, hypotheses):
            meteorScores.append(max([meteor_score(ref, hyp) for ref in refs]))
        meteor = np.mean(meteorScores)
        
        allResults[name] = {
            'bleu4': bleu4,
            'meteor': meteor,
            'time_s': elapsed,
            'type': 'LSTM',
        }
        print(f"{name:<22} BLEU-4={bleu4:.4f} METEOR={meteor:.4f} waktu={elapsed:.1f}s")
        
    except Exception as e:
        print(f"Gagal di {name}: {e}")


Loop evaluasi mengukur performa semua 12 variasi decoder (6 RNN + 6 LSTM):
- **BLEU-4**: metrik n-gram untuk kecocokan kata
- **METEOR**: metrik semantik yang mempertimbangkan bentuk kata dan sinonim

Langkah umum:
1. Buat caption untuk setiap gambar (greedy decoding)
2. Bandingkan dengan 5 caption referensi
3. Hitung BLEU-4 tingkat korpus
4. Hitung METEOR rata-rata per gambar

**Catatan:**
- Bobot BLEU-4 (0.25, 0.25, 0.25, 0.25) seimbang dari 1-gram sampai 4-gram
- METEOR lebih peka pada makna, bukan hanya kata yang persis sama
- Waktu inferensi dicatat untuk membandingkan efisiensi
- Ekspektasi: LSTM biasanya lebih baik dari RNN

In [ ]:
print("\n" + "="*70)
print("RINGKASAN HASIL")
print("="*70)
print(f"\n{'Model':<22} {'Tipe':<6} {'BLEU-4':>10} {'METEOR':>10} {'Waktu(dtk)':>11}")
print("-" * 70)

for name in sorted(allResults.keys()):
    res = allResults[name]
    print(f"{name:<22} {res['type']:<6} {res['bleu4']:>10.4f} {res['meteor']:>10.4f} {res['time_s']:>11.1f}")

bestByBleu = max(allResults, key=lambda n: allResults[n]['bleu4'])
bestByMeteor = max(allResults, key=lambda n: allResults[n]['meteor'])
bestRnn = max([n for n in allResults if allResults[n]['type'] == 'RNN'],
              key=lambda n: allResults[n]['bleu4'])
bestLstm = max([n for n in allResults if allResults[n]['type'] == 'LSTM'],
               key=lambda n: allResults[n]['bleu4'])

print(f"\n{'='*70}")
print(f"Terbaik BLEU-4 : {bestByBleu} ({allResults[bestByBleu]['bleu4']:.4f})")
print(f"Terbaik METEOR : {bestByMeteor} ({allResults[bestByMeteor]['meteor']:.4f})")
print(f"Terbaik RNN    : {bestRnn} ({allResults[bestRnn]['bleu4']:.4f})")
print(f"Terbaik LSTM   : {bestLstm} ({allResults[bestLstm]['bleu4']:.4f})")
print(f"{'='*70}")

rnnResults = {n: r for n, r in allResults.items() if r['type'] == 'RNN'}
lstmResults = {n: r for n, r in allResults.items() if r['type'] == 'LSTM'}

avgRnnBleu = np.mean([r['bleu4'] for r in rnnResults.values()])
avgLstmBleu = np.mean([r['bleu4'] for r in lstmResults.values()])
avgRnnMeteor = np.mean([r['meteor'] for r in rnnResults.values()])
avgLstmMeteor = np.mean([r['meteor'] for r in lstmResults.values()])

print(f"\nPerbandingan rata-rata (6 variasi):")
print(f"  RNN : BLEU-4={avgRnnBleu:.4f}, METEOR={avgRnnMeteor:.4f}")
print(f"  LSTM: BLEU-4={avgLstmBleu:.4f}, METEOR={avgLstmMeteor:.4f}")
print(f"  Selisih BLEU-4: {(avgLstmBleu - avgRnnBleu):.4f} ({(avgLstmBleu - avgRnnBleu)/avgRnnBleu*100:.1f}%)")
print(f"  Selisih METEOR: {(avgLstmMeteor - avgRnnMeteor):.4f} ({(avgLstmMeteor - avgRnnMeteor)/avgRnnMeteor*100:.1f}%)")

### Ringkasan dan Perbandingan RNN vs LSTM

Ringkasan ini menampilkan peringkat semua 12 model berdasarkan BLEU-4.
1. **Terbaik keseluruhan**: model dengan BLEU-4 tertinggi
2. **Terbaik RNN vs LSTM**: membandingkan dua arsitektur
3. **Rata-rata performa**: RNN vs LSTM pada 6 variasi masing-masing

**Ekspektasi umum:**
- LSTM cenderung lebih baik karena mekanisme cell state
- Layer/hidden lebih besar biasanya meningkatkan kualitas, tapi ada titik jenuh
- LSTM lebih lambat, namun akurasi biasanya lebih tinggi
- Peningkatan BLEU-4 dari RNN ke LSTM biasanya terlihat nyata

In [ ]:
# Visualisasi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Perbandingan BLEU-4
ax = axes[0, 0]
models = sorted(allResults.keys())
bleus = [allResults[m]['bleu4'] for m in models]
types = ['RNN' if allResults[m]['type'] == 'RNN' else 'LSTM' for m in models]
colors = ['#1f77b4' if t == 'RNN' else '#ff7f0e' for t in types]

ax.barh(range(len(models)), bleus, color=colors)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Skor BLEU-4')
ax.set_title('Skor BLEU-4: 12 Variasi')
ax.grid(axis='x', alpha=0.3)

# Plot 2: Perbandingan METEOR
ax = axes[0, 1]
meteors = [allResults[m]['meteor'] for m in models]
ax.barh(range(len(models)), meteors, color=colors)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Skor METEOR')
ax.set_title('Skor METEOR: 12 Variasi')
ax.grid(axis='x', alpha=0.3)

# Plot 3: Distribusi BLEU-4 per arsitektur
ax = axes[1, 0]
x = ['RNN', 'LSTM']
rnn_bleus = [allResults[n]['bleu4'] for n in rnnResults]
lstm_bleus = [allResults[n]['bleu4'] for n in lstmResults]

positions = [0, 1]
bp = ax.boxplot([rnn_bleus, lstm_bleus], positions=positions, widths=0.6, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#1f77b4', '#ff7f0e']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.scatter([0]*len(rnn_bleus), rnn_bleus, alpha=0.5, s=50, color='#1f77b4')
ax.scatter([1]*len(lstm_bleus), lstm_bleus, alpha=0.5, s=50, color='#ff7f0e')

ax.set_xticklabels(x)
ax.set_ylabel('BLEU-4')
ax.set_title('Distribusi BLEU-4: RNN vs LSTM')
ax.grid(axis='y', alpha=0.3)

# Plot 4: Waktu inferensi
ax = axes[1, 1]
times = [allResults[m]['time_s'] for m in models]
ax.barh(range(len(models)), times, color=colors)
ax.set_yticks(range(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel('Waktu inferensi (detik)')
ax.set_title('Waktu Inferensi pada Data Uji')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "src/wajib/weights/pipeline_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("Visualisasi tersimpan")

### Ringkasan Visualisasi 4 Panel

**Panel 1 (kiri atas): Skor BLEU-4** — bar horizontal untuk 12 model, biru=RNN, oranye=LSTM.
**Panel 2 (kanan atas): Skor METEOR** — metrik semantik yang melengkapi BLEU.
**Panel 3 (kiri bawah): Box plot RNN vs LSTM** — sebaran BLEU-4 tiap arsitektur.
**Panel 4 (kanan bawah): Waktu inferensi** — membandingkan biaya komputasi.

**Inti:** LSTM biasanya unggul pada kualitas, namun butuh waktu inferensi lebih lama.

In [ ]:
print("\n" + "="*70)
print("VARIASI PANJANG MAKSIMUM CAPTION (MODEL LSTM TERBAIK)")
print("="*70)

maxLenResults = {}

bestLstmPath = LSTM_WEIGHTS_DIR / f"{bestLstm}.keras"
model = tf.keras.models.load_model(str(bestLstmPath))

embed = EmbeddingLayer()
embed.loadWeights(model.get_layer('embedding'))

proj = DenseLayer()
proj.loadWeights(model.get_layer('cnn_proj'))

out = DenseLayer(activation='softmax')
out.loadWeights(model.get_layer('output'))

numLayersBest = int(bestLstm.split('L')[0].split('_')[1])
lstm = LSTMScratch()
lstm.loadWeights([model.get_layer(f'lstm_{i}') for i in range(numLayersBest)])

# Uji beberapa nilai max_len
maxLenVariations = [10, 20, 30, 40, 50]

for maxLenTest in maxLenVariations:
    hypotheses = []
    references = []
    
    for imgName, caps in testCaps.items():
        if imgName not in imageFeatures:
            continue
        
        feat = imageFeatures[imgName]
        
        # Buat caption dengan max_len yang berbeda
        x = proj.forward(feat)
        h = [np.zeros(cell.hiddenDim) for cell in lstm.cells]
        c = [np.zeros(cell.hiddenDim) for cell in lstm.cells]
        
        words = []
        token = vocab['<start>']
        
        for _ in range(maxLenTest):
            x_emb = embed.forward(token)
            h, c = lstm.forwardStep(x_emb, h, c)
            logits = out.forward(h[-1])
            token = int(np.argmax(logits))
            
            if token == vocab['<end>']:
                break
            words.append(id2word.get(token, '<unk>'))
        
        hyp = ' '.join(words)
        hypotheses.append(hyp.split())
        references.append([cap.split() for cap in caps])
    
    # Hitung BLEU-4
    bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25))
    maxLenResults[maxLenTest] = bleu4
    print(f"  max_len={maxLenTest:2d}  ->  BLEU-4={bleu4:.4f}")

# Plot max_len vs BLEU-4
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
maxLens = sorted(maxLenResults.keys())
bleus = [maxLenResults[ml] for ml in maxLens]

ax.plot(maxLens, bleus, marker='o', linewidth=2, markersize=8, color='#ff7f0e')
ax.fill_between(maxLens, bleus, alpha=0.3, color='#ff7f0e')
ax.set_xlabel('Panjang Maksimum Caption')
ax.set_ylabel('Skor BLEU-4')
ax.set_title(f'Pengaruh Panjang Maksimum Caption terhadap BLEU-4\n(Model LSTM terbaik: {bestLstm})')
ax.grid(True, alpha=0.3)
ax.set_xticks(maxLens)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "src/wajib/weights/maxlen_variation.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPanjang maksimum terbaik: {max(maxLenResults, key=maxLenResults.get)} (BLEU-4={max(maxLenResults.values()):.4f})")


### Eksplorasi: Pengaruh Panjang Maksimum Caption

Menggunakan model LSTM terbaik, kita uji beberapa nilai max_len: [10, 20, 30, 40, 50].
Tujuannya melihat trade-off antara caption yang terlalu pendek dan terlalu panjang.

**Pola yang umum terjadi:**
- max_len kecil -> caption terpotong, skor BLEU rendah
- max_len sedang (30-40) -> hasil paling stabil
- max_len besar -> caption terlalu panjang, skor bisa turun

**Catatan:**
- Rata-rata caption Flickr8k sekitar 11-12 kata
- max_len=30 sudah cukup aman untuk training
- Uji ini membantu memastikan model tidak terlalu sensitif terhadap batas panjang

## Kesimpulan Notebook 5: Evaluasi Pipeline Penuh

**Temuan utama:**
1. **LSTM lebih baik dari RNN** dalam kualitas caption
2. **Kedalaman optimal**: 2-3 layer dengan hidden 256-512 biasanya paling seimbang
3. **Model terbaik** sering berada di variasi LSTM dengan hidden besar
4. **Biaya inferensi** LSTM lebih tinggi, tetapi masih layak untuk evaluasi offline
5. **Panjang caption** optimal berada di kisaran 30-40

**Interpretasi metrik:**
- BLEU-4 sensitif terhadap kecocokan kata
- METEOR lebih peka pada makna
- Keduanya saling melengkapi

**Validasi:**
- LSTM scratch mampu meniru model Keras
- Greedy decoding sudah cukup untuk evaluasi awal
- CNN encoder memberi sinyal awal yang kuat

**Langkah berikutnya (Notebook 6):** Analisis kualitatif pada contoh gambar dari tiap kuartil BLEU.